# 01 — Data Prep & Merge (Boxing Winner v1)

Goal: load raw CSVs, clean basic fields, standardize fighter names, merge fights + fighters.
Output: a merged DataFrame ready for EDA/feature engineering.


In [1]:
# Minimal deps
import os, re, json
import pandas as pd
import numpy as np

# Where to look for raw files.
# Option A: place CSVs locally under repo:
#   data/raw/fighters.csv
#   data/raw/popular_matches.csv
LOCAL_RAW = "data/raw"

# Option B (Colab): Google Drive fallback – comment out if not using Colab
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def find_paths():
    f_local = os.path.join(LOCAL_RAW, "fighters.csv")
    m_local = os.path.join(LOCAL_RAW, "popular_matches.csv")
    if os.path.exists(f_local) and os.path.exists(m_local):
        return f_local, m_local

    if IN_COLAB:
        drive.mount('/content/drive', force_remount=False)
        # Try common MyDrive locations
        candidates = [
            ("/content/drive/MyDrive/data/fighters.csv",
             "/content/drive/MyDrive/data/popular_matches.csv"),
            ("/content/drive/MyDrive/fighters.csv",
             "/content/drive/MyDrive/popular_matches.csv")
        ]
        for f, m in candidates:
            if os.path.exists(f) and os.path.exists(m):
                return f, m

    raise FileNotFoundError(
        "Could not find fighters.csv and popular_matches.csv.\n"
        "Place them under data/raw/ or in Google Drive (see code)."
    )

fighters_path, matches_path = find_paths()
fighters_path, matches_path


Mounted at /content/drive


('/content/drive/MyDrive/fighters.csv',
 '/content/drive/MyDrive/popular_matches.csv')

In [2]:
fighters = pd.read_csv(fighters_path)
matches  = pd.read_csv(matches_path)

display(fighters.head())
display(matches.head())
fighters.info()
matches.info()


,name,wins,looses,draws,ko_rate,stance,age,height,reach,country
0,Azizbek Abdumuxtar Abdugofurov,0,0,0,0%,Orthodox,Unknown,Unknown,Unknown,Uzbekistan
1,Franco Fernando Altamiranda,0,0,0,0%,Orthodox,Unknown,Unknown,Unknown,Argentina
2,Joaquin Saul Alvarez,0,0,0,0%,Orthodox,Unknown,Unknown,Unknown,"Venezuela, Bolivarian Republic of"
3,Saul Alvarez,54,1,2,63.2%,Orthodox,32,5.74 ft (1.75 m),70.47 inches (179 cm),Mexico
4,Sukru Altay,0,0,0,0%,Orthodox,38,Unknown,Unknown,Germany


,date,place,opponent_1,opponent_2,opponent_1_estimated_punch_power,opponent_2_estimated_punch_power,opponent_1_estimated_punch_resistance,opponent_2_estimated_punch_resistance,opponent_1_estimated_ability_to_take_punch,opponent_2_estimated_ability_to_take_punch,opponent_1_rounds_boxed,opponent_2_rounds_boxed,opponent_1_round_ko_percentage,opponent_2_round_ko_percentage,opponent_1_has_been_ko_percentage,opponent_2_has_been_ko_percentage,opponent_1_avg_weight,opponent_2_avg_weight,verdict
0,31 August 2019,Unknown,Vasyl Lomachenko,Luke Campbell,72,72,73.9,60.5,78.0,73.0,119,130.0,8.40,12.31,0.00,0.00,127.11,135.24,Lomachenko won via UD in round 12
1,19 September 2019,Unknown,Orlando Fiordigiglio,Sam Eggington,59,59,55.4,54.5,68.0,66.0,200,177.0,6.50,8.47,3.03,6.25,152.50,148.93,Eggington won via KO in round 2
2,5 October 2019,Unknown,Gennady Golovkin,Sergiy Derevyanchenko,82,82,68.0,63.3,84.0,75.0,200,81.0,17.50,12.35,0.00,0.00,159.66,164.04,Golovkin won via UD in round 12
3,12 October 2019,Unknown,Chazz Witherspoon,Oleksandr Usyk,69,69,59.6,68.9,79.0,80.0,182,125.0,15.93,9.60,4.76,NaN,292.71,200.10,Usyk won via RTD in round 7
4,27 September 2019,Unknown,Ebenezer Tetteh,Daniel Dubois,46,46,NaN,44.4,50.0,69.0,38,39.0,23.68,30.77,0.00,0.00,178.67,230.38,Dubois won via TKO in round 1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2760 entries, 0 to 2759
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   name     2760 non-null   object
 1   wins     2760 non-null   int64 
 2   looses   2760 non-null   int64 
 3   draws    2760 non-null   int64 
 4   ko_rate  2760 non-null   object
 5   stance   2760 non-null   object
 6   age      2760 non-null   object
 7   height   2760 non-null   object
 8   reach    2760 non-null   object
 9   country  2760 non-null   object
dtypes: int64(3), object(7)
memory usage: 215.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 19 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   date                                        152 non-null    object 
 1   place                                       152 non-null    object 


In [3]:
# fix typos, strip symbols, coerce numbers
fighters = fighters.rename(columns={"looses": "losses"})

fighters["ko_rate"] = (
    fighters["ko_rate"].astype(str).str.replace("%","", regex=False).astype(float).fillna(0.0)
)

def to_num(x):
    try:
        return pd.to_numeric(re.findall(r"[-+]?\d*\.?\d+", str(x))[0])
    except Exception:
        return np.nan

fighters["age"] = fighters["age"].apply(lambda x: pd.to_numeric(x, errors="coerce"))
fighters["height_ft"] = fighters["height"].apply(to_num)
fighters["reach_in"]  = fighters["reach"].apply(to_num)

# normalize names
def clean_name(s):
    s = str(s).lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

fighters["clean_name"] = fighters["name"].apply(clean_name)
fighters_clean = fighters[[
    "clean_name","name","wins","losses","draws","ko_rate","stance","age",
    "height_ft","reach_in","country"
]]
fighters_clean.head()


,clean_name,name,wins,losses,draws,ko_rate,stance,age,height_ft,reach_in,country
0,azizbek abdumuxtar abdugofurov,Azizbek Abdumuxtar Abdugofurov,0,0,0,0.0,Orthodox,NaN,NaN,NaN,Uzbekistan
1,franco fernando altamiranda,Franco Fernando Altamiranda,0,0,0,0.0,Orthodox,NaN,NaN,NaN,Argentina
2,joaquin saul alvarez,Joaquin Saul Alvarez,0,0,0,0.0,Orthodox,NaN,NaN,NaN,"Venezuela, Bolivarian Republic of"
3,saul alvarez,Saul Alvarez,54,1,2,63.2,Orthodox,32.0,5.74,70.47,Mexico
4,sukru altay,Sukru Altay,0,0,0,0.0,Orthodox,38.0,NaN,NaN,Germany


In [4]:
matches = matches.copy()
matches["date"] = pd.to_datetime(matches["date"], errors="coerce")

def extract_winner(verdict):
    s = str(verdict)
    # get the first token before ' won'
    m = re.match(r"([A-Za-z .'-]+)\s+won\b", s)
    return m.group(1).strip() if m else np.nan

matches["winner"] = matches["verdict"].apply(extract_winner)

# clean opponent names
matches["opponent_1_clean"] = matches["opponent_1"].apply(clean_name)
matches["opponent_2_clean"] = matches["opponent_2"].apply(clean_name)

# map winner to side (1 or 2) where possible
def winner_side(row):
    w = clean_name(row["winner"]) if pd.notna(row["winner"]) else None
    if not w: return np.nan
    if w == row["opponent_1_clean"]: return 1.0
    if w == row["opponent_2_clean"]: return 2.0
    return np.nan

matches["winner_side"] = matches.apply(winner_side, axis=1)
matches[["date","opponent_1","opponent_2","winner","winner_side"]].head()


,date,opponent_1,opponent_2,winner,winner_side
0,2019-08-31,Vasyl Lomachenko,Luke Campbell,Lomachenko,NaN
1,2019-09-19,Orlando Fiordigiglio,Sam Eggington,Eggington,NaN
2,2019-10-05,Gennady Golovkin,Sergiy Derevyanchenko,Golovkin,NaN
3,2019-10-12,Chazz Witherspoon,Oleksandr Usyk,Usyk,NaN
4,2019-09-27,Ebenezer Tetteh,Daniel Dubois,Dubois,NaN


In [5]:
# left-join twice: opponent_1 and opponent_2
m1 = matches.merge(
    fighters_clean.add_suffix("_1"),
    left_on="opponent_1_clean", right_on="clean_name_1", how="left"
).drop(columns=["clean_name_1"])

m2 = m1.merge(
    fighters_clean.add_suffix("_2"),
    left_on="opponent_2_clean", right_on="clean_name_2", how="left"
).drop(columns=["clean_name_2"])

merged = m2
print("Merged shape:", merged.shape)
merged.head()


Merged shape: (152, 43)


,date,place,opponent_1,opponent_2,opponent_1_estimated_punch_power,opponent_2_estimated_punch_power,opponent_1_estimated_punch_resistance,opponent_2_estimated_punch_resistance,opponent_1_estimated_ability_to_take_punch,opponent_2_estimated_ability_to_take_punch,...,name_2,wins_2,losses_2,draws_2,ko_rate_2,stance_2,age_2,height_ft_2,reach_in_2,country_2
0,2019-08-31,Unknown,Vasyl Lomachenko,Luke Campbell,72,72,73.9,60.5,78.0,73.0,...,luke campbell,20.0,3.0,0.0,66.7,Southpaw,34.0,5.74,70.87,United Kingdom
1,2019-09-19,Unknown,Orlando Fiordigiglio,Sam Eggington,59,59,55.4,54.5,68.0,66.0,...,sam eggington,28.0,7.0,0.0,48.6,Orthodox,28.0,5.91,NaN,United Kingdom
2,2019-10-05,Unknown,Gennady Golovkin,Sergiy Derevyanchenko,82,82,68.0,63.3,84.0,75.0,...,sergiy derevyanchenko,13.0,2.0,0.0,66.7,Orthodox,36.0,5.58,67.32,United States
3,2019-10-12,Unknown,Chazz Witherspoon,Oleksandr Usyk,69,69,59.6,68.9,79.0,80.0,...,oleksandr usyk,18.0,0.0,0.0,72.2,Southpaw,35.0,6.23,NaN,Ukraine
4,2019-09-27,Unknown,Ebenezer Tetteh,Daniel Dubois,46,46,NaN,44.4,50.0,69.0,...,daniel dubois,15.0,1.0,0.0,87.5,Orthodox,25.0,6.36,NaN,United Kingdom


In [6]:
# Save a light processed file for the next notebook (ignored by .gitignore)
os.makedirs("data/processed", exist_ok=True)
out_path = "data/processed/fights_merged.parquet"
merged.to_parquet(out_path, index=False)
out_path


'data/processed/fights_merged.parquet'